# Lab Assignment 1: Comparative Analysis of Probabilistic Primality Testing Algorithms

**UID:** 2024300251  
**Algorithms:** Fermat, Euler and Miller–Rabin

Run the cells from top to bottom. Keep `primality_dataset_1000.csv` in the same folder as this notebook. The dataset's `Actual_Type` column is used only for evaluation, never during testing.

## Cell 1: Import required libraries

In [ ]:
import pandas as pd
import random
import math
import time
import matplotlib.pyplot as plt
from pathlib import Path

## Cell 2: Generate UID-based bases

In [ ]:
UID = "2024300251"
digits = [int(d) for d in UID]

base1 = (int(UID[-2:]) % 97) + 2
base2 = (sum(digits) % 97) + 2
base3 = ((int(UID[:4]) + int(UID[-4:])) % 97) + 2
base4 = (sum(digits[0::2]) % 97) + 2
base5 = (sum(digits[1::2]) % 97) + 2

INITIAL_BASES = [base1, base2, base3, base4, base5]
print("UID:", UID)
print("Initial bases:", INITIAL_BASES)

## Cell 3: Select five valid bases for each number
A base must satisfy `2 <= base < n` and `gcd(base, n) == 1`. Invalid UID-derived bases are replaced randomly. The seed makes the replacements reproducible.

In [ ]:
def select_valid_bases(n, initial_bases, rng):
    # The primality of 2 and 3 is handled directly by the test functions.
    if n <= 3:
        return []

    selected = []
    for base in initial_bases:
        if 2 <= base < n and math.gcd(base, n) == 1:
            selected.append(base)
        else:
            valid_base = None
            while valid_base is None:
                candidate = rng.randrange(2, n)
                if math.gcd(candidate, n) == 1:
                    valid_base = candidate
            selected.append(valid_base)

    return selected

## Cell 4: Fermat Primality Test

In [ ]:
def fermat_test(n, bases):
    if n < 2:
        return "Composite"
    if n in (2, 3):
        return "Prime"
    if n % 2 == 0:
        return "Composite"

    for a in bases:
        if pow(a, n - 1, n) != 1:
            return "Composite"
    return "Prime"

## Cell 5: Jacobi Symbol

In [ ]:
def jacobi_symbol(a, n):
    if n <= 0 or n % 2 == 0:
        raise ValueError("n must be a positive odd integer")

    a %= n
    result = 1

    while a != 0:
        while a % 2 == 0:
            a //= 2
            if n % 8 in (3, 5):
                result = -result

        a, n = n, a

        if a % 4 == 3 and n % 4 == 3:
            result = -result

        a %= n

    return result if n == 1 else 0

## Cell 6: Euler Primality Test

In [ ]:
def euler_test(n, bases):
    if n < 2:
        return "Composite"
    if n in (2, 3):
        return "Prime"
    if n % 2 == 0:
        return "Composite"

    for a in bases:
        symbol = jacobi_symbol(a, n)
        if symbol == 0:
            return "Composite"
        if pow(a, (n - 1) // 2, n) != symbol % n:
            return "Composite"
    return "Prime"

## Cell 7: Miller-Rabin Primality Test

In [ ]:
def miller_rabin_test(n, bases):
    if n < 2:
        return "Composite"
    if n in (2, 3):
        return "Prime"
    if n % 2 == 0:
        return "Composite"

    # Write n - 1 as d * 2^s, where d is odd.
    d = n - 1
    s = 0
    while d % 2 == 0:
        d //= 2
        s += 1

    for a in bases:
        x = pow(a, d, n)

        if x == 1 or x == n - 1:
            continue

        passed = False
        for _ in range(s - 1):
            x = pow(x, 2, n)
            if x == n - 1:
                passed = True
                break

        if not passed:
            return "Composite"

    return "Prime"

## Cell 8: Load the dataset

In [ ]:
dataset_path = Path("primality_dataset_1000.csv")
if not dataset_path.exists():
    raise FileNotFoundError(
        "Place primality_dataset_1000.csv in the same folder as this notebook."
    )

df = pd.read_csv(dataset_path)
print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df.head())

## Cell 9: Run all three tests and measure execution time
The `Actual_Type` column is not passed to any primality-testing function.

In [ ]:
if "Number" not in df.columns:
    raise KeyError("The dataset must contain a 'Number' column.")

rng = random.Random(2024300251)

fermat_results, euler_results, mr_results = [], [], []
fermat_times, euler_times, mr_times = [], [], []
used_bases_column = []

for number in df["Number"]:
    n = int(number)
    bases = select_valid_bases(n, INITIAL_BASES, rng)
    used_bases_column.append(",".join(map(str, bases)))

    start = time.perf_counter()
    fermat_result = fermat_test(n, bases)
    fermat_times.append((time.perf_counter() - start) * 1000)
    fermat_results.append(fermat_result)

    start = time.perf_counter()
    euler_result = euler_test(n, bases)
    euler_times.append((time.perf_counter() - start) * 1000)
    euler_results.append(euler_result)

    start = time.perf_counter()
    mr_result = miller_rabin_test(n, bases)
    mr_times.append((time.perf_counter() - start) * 1000)
    mr_results.append(mr_result)

df["Bases_Used"] = used_bases_column
df["Fermat_Result"] = fermat_results
df["Fermat_Time_ms"] = fermat_times
df["Euler_Result"] = euler_results
df["Euler_Time_ms"] = euler_times
df["MillerRabin_Result"] = mr_results
df["MillerRabin_Time_ms"] = mr_times

display(df.head(10))

## Cell 10: Prepare actual labels for evaluation

In [ ]:
if "Actual_Type" not in df.columns:
    raise KeyError("The dataset must contain an 'Actual_Type' column.")

def actual_class(value):
    label = str(value).strip().lower()
    if "prime" in label and "composite" not in label and "carmichael" not in label:
        return "Prime"
    return "Composite"

df["Actual_Class"] = df["Actual_Type"].apply(actual_class)

## Cell 11: Calculate performance metrics

In [ ]:
def calculate_metrics(actual, predicted, times):
    correct = sum(a == p for a, p in zip(actual, predicted))
    incorrect = len(actual) - correct

    false_positives = sum(
        a == "Composite" and p == "Prime"
        for a, p in zip(actual, predicted)
    )
    false_negatives = sum(
        a == "Prime" and p == "Composite"
        for a, p in zip(actual, predicted)
    )

    return {
        "Correct": correct,
        "Incorrect": incorrect,
        "False Positives": false_positives,
        "False Negatives": false_negatives,
        "Accuracy (%)": correct / len(actual) * 100,
        "Avg. Time (ms)": sum(times) / len(times)
    }

actual = df["Actual_Class"].tolist()

summary = pd.DataFrame({
    "Fermat": calculate_metrics(actual, df["Fermat_Result"].tolist(), fermat_times),
    "Euler": calculate_metrics(actual, df["Euler_Result"].tolist(), euler_times),
    "Miller-Rabin": calculate_metrics(actual, df["MillerRabin_Result"].tolist(), mr_times)
}).T

display(summary.round(6))

## Cell 12: Display the detailed result table

In [ ]:
detailed_columns = [
    "Number", "Actual_Type",
    "Fermat_Result", "Euler_Result", "MillerRabin_Result",
    "Fermat_Time_ms", "Euler_Time_ms", "MillerRabin_Time_ms",
    "Bases_Used"
]
display(df[detailed_columns])

## Cell 13: Save the completed dataset and summary table

In [ ]:
df.to_csv("primality_dataset_1000_completed.csv", index=False)
summary.to_csv("primality_summary.csv")
print("Saved primality_dataset_1000_completed.csv")
print("Saved primality_summary.csv")

## Cell 14: Observation 1, which algorithm produced the highest accuracy?

In [ ]:
highest_accuracy = summary["Accuracy (%)"].max()
best_algorithms = summary.index[
    summary["Accuracy (%)"] == highest_accuracy
].tolist()

print("Highest accuracy:", highest_accuracy, "%")
print("Algorithm(s):", ", ".join(best_algorithms))

## Cell 15: Observation 2, which algorithm required the least execution time?

In [ ]:
least_time = summary["Avg. Time (ms)"].min()
fastest_algorithms = summary.index[
    summary["Avg. Time (ms)"] == least_time
].tolist()

print("Least average execution time:", least_time, "ms")
print("Algorithm(s):", ", ".join(fastest_algorithms))

## Cell 16: Observation 3, how many Carmichael numbers were incorrectly identified by Fermat's test?

In [ ]:
carmichael_mask = df["Actual_Type"].astype(str).str.contains(
    "Carmichael", case=False, na=False
)
fermat_carmichael_false_positives = (
    carmichael_mask & (df["Fermat_Result"] == "Prime")
).sum()

print(
    "Carmichael numbers incorrectly identified as Prime by Fermat:",
    int(fermat_carmichael_false_positives)
)

## Cell 17: Observation 4, did Euler's test improve upon Fermat's results?

In [ ]:
fermat_accuracy = summary.loc["Fermat", "Accuracy (%)"]
euler_accuracy = summary.loc["Euler", "Accuracy (%)"]

if euler_accuracy > fermat_accuracy:
    print("Yes. Euler's test improved accuracy by",
          round(euler_accuracy - fermat_accuracy, 4), "percentage points.")
elif euler_accuracy < fermat_accuracy:
    print("No. Euler's test had lower accuracy by",
          round(fermat_accuracy - euler_accuracy, 4), "percentage points.")
else:
    print("Euler's test and Fermat's test had the same accuracy.")

print("Fermat accuracy:", round(fermat_accuracy, 4), "%")
print("Euler accuracy:", round(euler_accuracy, 4), "%")

## Cell 18: Observation 5, why is Miller-Rabin preferred in practical cryptographic applications?

In [ ]:
print(
    "Miller-Rabin is preferred because it is efficient and has a much lower "
    "probability of falsely classifying a composite number as prime. Each "
    "independent round reduces the error probability, with an upper bound "
    "of 1/4 for an odd composite number per randomly selected base. It is "
    "therefore practical for testing the large candidate primes used in RSA."
)

## Cell 19: Observation 6, how does the choice of base influence the outcome?

In [ ]:
print(
    "The base determines the modular arithmetic check performed by each test. "
    "A base that exposes a compositeness witness makes the algorithm return "
    "Composite. Some bases can fail to expose certain composite numbers, "
    "causing a false positive. Testing with multiple suitable bases reduces "
    "this risk. The selected bases must be valid for the number being tested."
)

## Cell 20: Graph of accuracy

In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(summary.index, summary["Accuracy (%)"])
plt.ylabel("Accuracy (%)")
plt.xlabel("Algorithm")
plt.title("Accuracy Comparison of Primality Tests")
plt.ylim(0, 100)
plt.tight_layout()
plt.show()

## Cell 21: Graph of average execution time

In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(summary.index, summary["Avg. Time (ms)"])
plt.ylabel("Average Execution Time (ms)")
plt.xlabel("Algorithm")
plt.title("Average Execution Time Comparison")
plt.tight_layout()
plt.show()

## Cell 22: Graph of false positives

In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(summary.index, summary["False Positives"])
plt.ylabel("Number of False Positives")
plt.xlabel("Algorithm")
plt.title("False Positive Comparison")
plt.tight_layout()
plt.show()

## Cell 23: One-page conclusion

In [ ]:
print(f"""
CONCLUSION

The experiment compared Fermat, Euler and Miller-Rabin probabilistic
primality tests on the supplied dataset of 1000 numbers. Each number was
tested using five valid bases derived initially from UID {UID}. Invalid
bases were replaced with randomly generated valid bases.

The measured results were:
{summary.round(6).to_string()}

Fermat's test is based on Fermat's little theorem. It is simple and
computationally efficient, but Carmichael numbers can satisfy the test
for every base coprime to the number. This can lead to false positives.

Euler's test uses Euler's criterion and the Jacobi symbol. It can detect
some composite numbers that pass Fermat's test, although it is also a
probabilistic test and is not free from false positives.

Miller-Rabin decomposes n - 1 into an odd component and a power of two,
then checks modular exponentiation conditions. Repeating the test with
multiple bases significantly reduces the probability of accepting a
composite number as prime. This combination of efficiency and reliability
makes it widely used in practical cryptographic applications, including
RSA prime-candidate generation.

Execution time depends on the implementation, input values, selected
bases and computer hardware. The measured accuracy and timing results
above provide the experimental comparison for this dataset.
""")